<a href="https://colab.research.google.com/github/GinnaGomez09/hello-world/blob/master/03_tokenizacion_lematizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 280, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 280 (delta 95), reused 18 (delta 18), pack-reused 142 (from 1)
Receiving objects: 100% (280/280), 3.65 MiB | 5.30 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/proyecto_aplicado_javeriana


In [2]:
# ============================================================
# 2. Importar librerías
# ============================================================

import pandas as pd
import numpy as np
import os
import re
import unicodedata

In [3]:
# ============================================================
# 3. Cargar dataset original
# ============================================================

recetas_path = "data/raw/recetas/recetas_ingredientes.csv"

recetas = pd.read_csv(recetas_path)

print("Dimensiones:", recetas.shape)

recetas.head()

Dimensiones: (31587, 16)


,receta_uuid,receta_titulo,receta_url,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,cantidad_min,cantidad_max,unidad,ingrediente_nombre,tcac_alimento_codigo,tcac_alimento_nombre,match_method,match_score,cantidad_gramos_est
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,NaN,NaN,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,NaN,NaN
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,2,1 taza de agua tibia,1,1.000000,NaN,NaN,taza,agua tibia,NaN,NaN,NaN,NaN,NaN
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,NaN,NaN,taza,queso mozzarella o queso blanco rallado,NaN,NaN,NaN,NaN,NaN
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,4,2 cucharadas de mantequilla,2,2.000000,NaN,NaN,cucharadas,mantequilla,NaN,NaN,NaN,NaN,NaN
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,5,Sal,NaN,NaN,NaN,NaN,NaN,Sal,NaN,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# 4. Definición de columnas clave
# ============================================================

col_linea = "ingrediente_linea"
col_ingrediente = "ingrediente_nombre"
col_unidad = "unidad"

print("Columnas utilizadas:")
print(col_linea)
print(col_ingrediente)
print(col_unidad)

Columnas utilizadas:
ingrediente_linea
ingrediente_nombre
unidad


In [5]:
# ============================================================
# 5. Funciones de limpieza de texto
# ============================================================

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    texto = unicodedata.normalize("NFKD", texto)

    texto = "".join(
        [c for c in texto if not unicodedata.combining(c)]
    )

    return texto


def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # minúsculas
    texto = texto.lower()

    # eliminar tildes
    texto = quitar_tildes(texto)

    # eliminar caracteres especiales
    texto = re.sub(r"[^a-z0-9\s\/\.,]", " ", texto)

    # normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [6]:
# ============================================================
# 6. Aplicar limpieza antes del NLP
# ============================================================

recetas["ingrediente_linea_limpia"] = recetas[col_linea].apply(
    limpiar_texto
)

recetas["ingrediente_nombre_limpio"] = recetas[col_ingrediente].apply(
    limpiar_texto
)

print("Limpieza aplicada correctamente")

recetas[
    [
        col_linea,
        "ingrediente_linea_limpia",
        col_ingrediente,
        "ingrediente_nombre_limpio"
    ]
].head()

Limpieza aplicada correctamente


,ingrediente_linea,ingrediente_linea_limpia,ingrediente_nombre,ingrediente_nombre_limpio
0,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,harina de arepa blanca o amarilla
1,1 taza de agua tibia,1 taza de agua tibia,agua tibia,agua tibia
2,⅓ taza de queso mozzarella o queso blanco rallado,1 3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,queso mozzarella o queso blanco rallado
3,2 cucharadas de mantequilla,2 cucharadas de mantequilla,mantequilla,mantequilla
4,Sal,sal,Sal,sal


In [7]:
# ============================================================
# 7. Definición de columnas para NLP
# ============================================================

col_linea_limpia = "ingrediente_linea_limpia"
col_ingrediente_limpio = "ingrediente_nombre_limpio"

print("Columnas NLP:")
print(col_linea_limpia)
print(col_ingrediente_limpio)

Columnas NLP:
ingrediente_linea_limpia
ingrediente_nombre_limpio


In [8]:
# ============================================================
# 8. Instalación y carga de spaCy
# ============================================================

!pip install spacy -q
!python -m spacy download es_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 69.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
import spacy

nlp = spacy.load("es_core_news_sm")

print("Modelo spaCy cargado correctamente")

Modelo spaCy cargado correctamente


In [10]:
# ============================================================
# 9. Prueba inicial de tokenización y lematización
# ============================================================

ejemplo = recetas[col_linea_limpia].dropna().iloc[0]

doc = nlp(ejemplo)

print("Texto original limpio:")
print(ejemplo)

print("\nTokens:")
print([token.text for token in doc])

print("\nLemas:")
print([token.lemma_ for token in doc])

print("\nPOS tags:")
print([(token.text, token.pos_) for token in doc])

Texto original limpio:
1 taza de harina de arepa blanca o amarilla

Tokens:
['1', 'taza', 'de', 'harina', 'de', 'arepa', 'blanca', 'o', 'amarilla']

Lemas:
['1', 'taza', 'de', 'harina', 'de', 'arepa', 'blanco', 'o', 'amarillo']

POS tags:
[('1', 'NUM'), ('taza', 'NOUN'), ('de', 'ADP'), ('harina', 'NOUN'), ('de', 'ADP'), ('arepa', 'PROPN'), ('blanca', 'ADJ'), ('o', 'CCONJ'), ('amarilla', 'ADJ')]
